# 🎯 03 - Indexing and Selection

Time to slice and dice.

In this notebook, you'll:
- Use `.loc[]` for label-based selection, and `.iloc[]` for position-based selection
- Create boolean masks to filter rows
- Select specific rows **and** columns together in one step
- Change values safely, without the "chained assignment" trap

## ✅ Learning Goals
- Tell `.loc[]` and `.iloc[]` apart, and know when to use each
- Select specific rows and columns with lists of labels or positions
- Give a DataFrame a meaningful index with `.set_index()`
- Grab a single value with `.at[]` and `.iat[]`
- Filter rows with a boolean mask, combine conditions with `&` and `|`, and flip one with `~`
- Match several values with `.isin()`, a range with `.between()`, and missing values with `.isna()`/`.notna()`
- Know when a selection gives you a `Series` and when it gives you a `DataFrame`
- Select rows and columns together with `df.loc[condition, columns]`, and change values without chained assignment

> **The core habit:** use `.loc[]` when the labels matter, `.iloc[]` when positions matter, and boolean masks when conditions matter.

In [ ]:
import pandas as pd

df = pd.read_csv('titanic.csv')
df.head(3)

Both selectors use the same shape: **rows first, then columns**, separated by a comma.

```python
df.loc[row_labels, column_labels]          # by label (names)
df.iloc[row_positions, column_positions]   # by position (0, 1, 2, ...)
```

Leave out the columns to get every column. A bare `:` also means "all", so `df.loc[:, ['Age']]` is every row of the `Age` column.

## 🔎 Select Rows by Label with `.loc[]`

In [ ]:
# .loc[row_label, column_label]
# Titanic uses the default RangeIndex, so labels here happen to be integers
print(df.loc[0])

In [ ]:
# Row range (inclusive on both ends!) with specific columns
df.loc[0:4, ['Age', 'Fare', 'Sex']]

To pick specific rows that aren't next to each other, pass a **list** of labels:

In [ ]:
df.loc[[0, 2, 4], ['Name', 'Age']]

## 🔢 Select Rows by Position with `.iloc[]`

In [ ]:
df.iloc[0]  # First row, by position

In [ ]:
# Row slice, column slice — both by position, stop-exclusive like a normal Python slice
df.iloc[1:4, 0:5]

`.iloc[]` takes lists too, of **positions**. Columns 3 and 5 are `Name` and `Age`:

In [ ]:
df.iloc[[0, 2, 4], [3, 5]]

### 🧠 Quick Check
`.loc[0:4]` includes row `4`. `.iloc[0:4]` does **not** include position `4`. Why do you think `.loc[]` breaks the usual "stop-exclusive" slicing rule you learned for lists and strings? (Hint: think about what happens when labels aren't numbers at all, like `.loc['Alice':'Charlie']`.)

## 🏷️ A Meaningful Index with `.set_index()`
Row labels `0, 1, 2, ...` don't mean much. `.set_index()` turns a column into the row labels, so `.loc[]` can look rows up by something meaningful. (`.iloc[]` is unaffected: it always uses positions.)

In [ ]:
by_id = df.set_index('PassengerId')
by_id.loc[2, ['Name', 'Age', 'Fare']]   # passenger #2, not the third row

In [ ]:
by_name = df.set_index('Name')
by_name.loc['Braund, Mr. Owen Harris', 'Age']

### When `.loc[]` and `.iloc[]` Disagree
With the default `0, 1, 2, ...` index, a row's label and its position are the same number, so `.loc[3]` and `.iloc[3]` look interchangeable. Once the index means something, they don't:

In [ ]:
print('by_id.loc[3]  →', by_id.loc[3, 'Name'])    # the row LABELED 3: passenger #3
print('by_id.iloc[3] →', by_id.iloc[3]['Name'])   # the row at POSITION 3: the 4th row

| | `.loc[]` | `.iloc[]` |
|---|---|---|
| Selects using | **Labels** (index values and column names) | **Positions** (`0, 1, 2, ...`) |
| Example | `df.loc[2, 'Fare']` | `df.iloc[2, 9]` |
| End of a slice | **Included** (`.loc[1:3]` gives 3 rows) | **Excluded** (`.iloc[1:3]` gives 2 rows) |
| Columns chosen by | Name | Position |
| Best for | Readable, named selections | "The first 5 rows", "the last column" |

`.set_index()` returns a **new** DataFrame, so `df` itself still has its original `0, 1, 2, ...` index.

## 🎯 Grabbing One Value: `.at[]` and `.iat[]`
`.loc[]` and `.iloc[]` can return a single value, but `.at[]` (by label) and `.iat[]` (by position) are made for exactly that, and are faster:

In [ ]:
print(df.loc[1, 'Fare'])   # label-based, general purpose
print(df.iloc[1, 9])       # position-based: row 1, column 9 (Fare)
print(df.at[1, 'Fare'])    # label-based, ONE value
print(df.iat[1, 9])        # position-based, ONE value

## 🎯 Boolean Masking (Filtering)

A condition on a column doesn't return rows. It returns a **mask**: a `Series` of `True`/`False`, one per row:

In [ ]:
under_10 = df['Age'] < 10
print(under_10.head(8))
print('How many are True:', under_10.sum())   # True counts as 1, False as 0

Putting the mask inside `df[...]` keeps only the rows where it's `True`. `df.loc[mask]` does exactly the same thing:

In [ ]:
# Filter passengers under age 10
kids = df[df['Age'] < 10]
kids[['Age', 'Sex', 'Pclass']].head()

## 🧠 Compound Conditions
Use `&` for "and" and `|` for "or" — **not** the `and`/`or` keywords from Control Flow. Each condition also needs its own parentheses.

In [ ]:
# Adults in First Class who paid over $100
rich_folks = df[(df['Age'] > 18) & (df['Pclass'] == 1) & (df['Fare'] > 100)]
rich_folks[['Age', 'Pclass', 'Fare']].head()

### Why Not `and` / `or`?
`df[(df['Age'] < 10) and (df['Pclass'] == 1)]` fails with:

```text
ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
```

Python's `and` wants **one** True or False, but a mask holds **hundreds** of them, and Python can't tell which one you mean. `&` and `|` combine two masks **row by row**, which is what you want. Forgetting the parentheses causes a similar error, because `&` is evaluated before `<` and `==`.

## 🧺 Matching Several Values: `.isin()` and `.between()`
Checking one column against a **list** of values with `|` gets long fast. `.isin()` does it in one step, and `.between()` checks a range. Both ends are **included**:

In [ ]:
# Instead of (df['Embarked'] == 'C') | (df['Embarked'] == 'Q')
cherbourg_or_queenstown = df[df['Embarked'].isin(['C', 'Q'])]
print('Embarked at C or Q:', len(cherbourg_or_queenstown))

teens = df[df['Age'].between(13, 19)]      # 13 <= Age <= 19
print('Teenagers:', len(teens))

### Flipping a Condition with `~`
`~` means **not**. It turns every `True` into `False` and every `False` into `True`. For a simple comparison, `!=` is just as good, but `~` is the only way to say "not" with `.isin()`, `.between()`, or `.isna()`:

In [ ]:
not_southampton = df[~df['Embarked'].isin(['S'])]
print('Did NOT board at Southampton:', len(not_southampton))

### Missing Values as a Mask
`.isna()` is `True` where a value is missing, and `.notna()` is the opposite:

In [ ]:
print('Age missing:', df['Age'].isna().sum())
print('Age known:  ', df['Age'].notna().sum())
df[df['Age'].isna()][['Name', 'Age']].head(3)

💡 Text columns have their own masks too, like `df['Name'].str.contains('Mrs')`. Notebook 06 covers those.

## 🧹 Select Columns Only

In [ ]:
df[['Sex', 'Age', 'Embarked']].head()

### Series or DataFrame?
**Single** brackets with one name give you a `Series`. **Double** brackets (a list of names, even a list of one) give you a `DataFrame`:

In [ ]:
print(type(df['Age']))      # Series
print(type(df[['Age']]))    # DataFrame (a table with one column)

You may also see `df.Age` (attribute notation). It works for simple names, but **prefer brackets**. They work for every column name, including ones with spaces (`df['order total']`) and ones that clash with DataFrame methods (a column named `count` or `size`).

## 🎯 Rows and Columns Together
The single most useful selection pattern combines a **mask for the rows** with a **list of columns**, in one `.loc[]`:

In [ ]:
df.loc[df['Pclass'] == 1, ['Name', 'Age', 'Fare']].head()

Read it as: *"From `df`, keep the rows where `Pclass` is 1, then return `Name`, `Age`, and `Fare`."* The row condition can be as complex as you need:

In [ ]:
df.loc[(df['Sex'] == 'female') & (df['Fare'] < 10), ['Name', 'Age', 'Fare']].head()

### The Top or Bottom N
To find the rows with the largest or smallest values in a column, `.nlargest()` and `.nsmallest()` are shortcuts for sorting and then taking `.head()`:

In [ ]:
df.nlargest(5, 'Fare')[['Name', 'Pclass', 'Fare']]

## ✏️ Changing Values: Avoid Chained Assignment
Selecting and then assigning in **two** bracket steps looks reasonable, but it doesn't work. The first step makes a **copy**, so the assignment changes the copy and then throws it away. Pandas warns you, but the original is untouched:

In [ ]:
# ❌ Two steps: filter, THEN assign to the filtered copy
df[df['Age'] < 10]['child'] = True

print('Did it add a "child" column?', 'child' in df.columns)

Do it in **one** `.loc[]` step instead. `.loc[]` names the original DataFrame, the rows, and the column all at once. Give every row a default first, so the rows that don't match aren't left as `NaN` (notebook 05 uses this pattern to build new columns):

In [ ]:
# ✅ One step, with .loc[rows, column]
df['child'] = False
df.loc[df['Age'] < 10, 'child'] = True

df['child'].value_counts()

💡 If you make a filtered DataFrame that you plan to **change**, add `.copy()`, as in `kids = df[df['Age'] < 10].copy()`. That makes it clear you mean a separate table, and pandas won't warn you.

## 🔢 Resetting the Index After Filtering
A filtered or sorted DataFrame keeps its **original** row labels, so they're no longer `0, 1, 2, ...`:

In [ ]:
kids = df[df['Age'] < 10]
print('Labels after filtering:', kids.index[:5].tolist())

kids = kids.reset_index(drop=True)
print('Labels after reset:    ', kids.index[:5].tolist())

`drop=True` throws the old labels away instead of keeping them as a new column. Don't reset out of habit: if the index **means** something (like `PassengerId`), resetting loses it.

## 🧭 A Selection Workflow
Before writing a selection, ask yourself:
1. Do I need **rows**, **columns**, or both?
2. Am I selecting by **label** or by **position**?
3. Do I need a **condition**?
4. Should the result be a `Series` or a `DataFrame`?
5. Am I **changing** the original data? (If so, use one `.loc[]` step.)

Most real selections end up looking like this: a condition, a column list, and maybe a sort:

In [ ]:
(df.loc[(df['Pclass'] == 3) & (df['Survived'] == 1), ['Name', 'Age', 'Fare']]
   .sort_values('Fare', ascending=False)
   .head())

## 🏋️ Practice

### Practice 1 — Basic Selection (easy)
Select the first row using `.iloc[]`, then again using `.loc[]`. Confirm they match.

In [ ]:
# Your code here


### Practice 2 — Filter and Select (medium)
Filter the dataset for female passengers over age 40, showing only the `Name`, `Age`, and `Pclass` columns.

In [ ]:
# Your code here


### Practice 3 — Missing Data Slice (harder)
Find passengers with a missing `Age` (hint: `.isnull()`) and print only their `Name`, `Age`, and `Pclass` columns.

In [ ]:
# Your code here


### Practice 4 — `.isin()` and `.between()` (medium)
Find passengers who embarked at Cherbourg (`'C'`) or Queenstown (`'Q'`) **and** paid a fare between \$50 and \$100. Show only `Name`, `Embarked`, and `Fare`. How many are there?

In [ ]:
# Your code here


### Practice 5 — Not and Not Missing (medium)
Using `~` and `.notna()`, find passengers who did **not** travel in 3rd class **and** whose `Age` is known. Show only `Name`, `Pclass`, and `Age` with a single `.loc[]`, then reset the index so the rows are numbered from 0.

In [ ]:
# Your code here


## ✏️ Your Turn
1. Slice the first 5 rows and select only `Survived`, `Sex`, and `Age`.
2. Create a DataFrame of passengers in 2nd class under age 15.
3. Use `.loc[]` to select rows 10–20 and only the columns `Fare`, `Pclass`, and `Age`. Then use `.iloc[]` to get the **same** rows and columns by position.
4. Make a copy indexed by `Name` with `.set_index()`, then use `.at[]` to look up one passenger's `Fare`.
5. Show the `Name` and `Age` of the 5 **youngest** passengers with `.nsmallest()`.
6. Add an `age_unknown` column that's `True` for passengers with a missing `Age` and `False` otherwise, using `.loc[]` (no chained assignment!).
7. In a comment, explain why `df.loc[1:3]` returns 3 rows but `df.iloc[1:3]` returns 2.

In [ ]:
# Your turn here


## 🔥 Challenge (Optional) — One Expression
Find the **10 youngest female survivors in 1st or 2nd class**, showing only `Name`, `Age`, and `Pclass`, sorted from youngest to oldest, with rows numbered `0`–`9`.

Do it as **one** chained expression: a `.loc[]` with a compound condition (use `.isin()` for the classes) and a column list, then `.sort_values()`, `.head()`, and `.reset_index(drop=True)`. Passengers with a missing `Age` shouldn't appear. Does `.sort_values()` put them last, or do you need to filter them out?

In [ ]:
# Your code here


---
## 🧼 Why This Matters
Being able to surgically select parts of your dataset is the core of real analysis. Whether you're cleaning junk or feeding a model, proper indexing is power.

The most flexible everyday pattern is `df.loc[row_condition, ['column_a', 'column_b']]`. It reads naturally, selects exactly what you need, and works for both looking at data and changing it.

Next: cleaning your data with `.dropna()`, `.fillna()`, and more.